In [1]:
import pandas as pd
import geopandas as gpd
import os

In [2]:
def get_id_attr(df, yr, months, suffix):
    df['ctyfips'] = df.ids.str[:5].astype(int)
    df['year'] = df.ids.str[-4:].astype(int)
    df = df[df.year==yr]
    df.columns = [ col + "_" + suffix if col in months else col
        for col in df.columns
    ]
    df = df.drop(columns=["ids"])
    df = df.set_index(['ctyfips', 'year'])
    return df

In [3]:
months = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

cols = ["ids"]+months

cty_crosswalk = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/county-to-climdivs.txt", sep='\s+')

df_precip = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-pcpncy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_precip["all"] = df_precip.loc[:, months].sum(axis=1)
df_temp_min = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-tmincy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_temp_min["all"] = df_temp_min.loc[:, months].min(axis=1)
df_temp_mean = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-tmpccy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_temp_mean["all"] = df_temp_mean.loc[:, months].mean(axis=1)
df_temp_max = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-tmaxcy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_temp_max["all"] = df_temp_max.loc[:, months].max(axis=1)
df_cdd = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-cddccy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_cdd["all"] = df_cdd.loc[:, months].sum(axis=1)
df_hdd = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/climdiv-hddccy-v1.0.0-20250107.txt", sep='\s+', names=cols, dtype={'ids': str})
df_hdd["all"] = df_hdd.loc[:, months].sum(axis=1)

months = months+["all"]

# Process first column to extract location and year details - filter for Vulcan year of 2019
df_precip = get_id_attr(df_precip, 2019, months, "prec")
df_temp_min = get_id_attr(df_temp_min, 2019, months, "tmin")
df_temp_mean = get_id_attr(df_temp_mean, 2019, months, "tmean")
df_temp_max = get_id_attr(df_temp_max, 2019, months, "tmax")
df_cdd = get_id_attr(df_cdd, 2019, months, "cdd")
df_hdd = get_id_attr(df_hdd, 2019, months, "hdd")

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:17: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_605355/3718008754.py:5: SyntaxWarning: invalid escape sequence '\s'
  cty_crosswalk = pd.read_csv("/work/hawkins_lab/vulcan/data/climate/county-to-climdivs.txt", sep='\s+')
/tmp/ipykernel_605355/3718008754.py:7: SyntaxWarning: invalid escape sequence '\s'
  df_precip = pd.read

In [4]:
# Join the dataframes into one
df_climate = df_precip.join(df_temp_min, on=['ctyfips', 'year'], how='left').join(df_temp_mean, on=['ctyfips', 'year'], how='left')\
            .join(df_temp_max, on=['ctyfips', 'year'], how='left').join(df_cdd, on=['ctyfips', 'year'], how='left').join(df_hdd, on=['ctyfips', 'year'], how='left').reset_index()

df_climate.to_csv("/work/hawkins_lab/vulcan/data/climate/climate_county_2019.csv")

add_cbg_stats = pd.read_csv('/work/hawkins_lab/vulcan/data/nhgis0021_ds239_20185_blck_grp_clean.csv', usecols=['geoid','p_highschool','p_owner','avg_hh_size','med_dwelling_age'])
add_cbg_stats['geoid20'] = add_cbg_stats['geoid'].str[7:].astype(int)

In [5]:
cty_crosswalk = dict(zip(cty_crosswalk.NCDC_FIPS_ID,cty_crosswalk.POSTAL_FIPS_ID))
df_climate.ctyfips = df_climate.ctyfips.map(cty_crosswalk)

In [6]:
df_fips = pd.read_csv('state_and_county_fips_master.csv')
df_climate = pd.merge(df_climate, df_fips, left_on="ctyfips",right_on="fips",how="right")

df_climate = df_climate.sort_values(by="fips", ascending=False)
df_climate = df_climate.bfill()
df_climate = df_climate.bfill()
df_climate.ctyfips = df_climate.fips

In [7]:
df_climate.reset_index(inplace=True)

In [8]:
base_path = '/work/hawkins_lab/vulcan/data/vulcan'

res_par_file = os.path.join(base_path,'parquet','vulcan_RES_epa')
com_par_file = os.path.join(base_path,'parquet','vulcan_COM_epa')
ind_par_file = os.path.join(base_path,'parquet','vulcan_IND_epa')
tran_par_file = os.path.join(base_path,'parquet','vulcan_ONR_epa')
tot_par_file = os.path.join(base_path,'parquet','vulcan_TOT_epa')

file_list = [res_par_file, com_par_file, ind_par_file, tran_par_file, tot_par_file]

In [9]:
# def modify_GISJOIN(n):
#     s = str(n)
#     # Remove the third and last digits (index 2 and -1)
#     return int(s[:2] + s[3:-1])

# df_climate.ctyfips = df_climate.GISJOIN.map(modify_GISJOIN)
# print(df_climate.GISJOIN)

We found that the climate data was missing a few county fips codes. Let's assume the climate is similar in adjacent counties.

In [10]:
df = gpd.read_parquet(tot_par_file + '.parquet')
df_climate = pd.merge(df_climate, df.loc[:, ['fips','cbsa']], on='fips', how='left')

In [11]:
df_climate = df_climate.groupby('cbsa').agg('mean').reset_index().drop(columns=['ctyfips','fips'])

In [12]:
df_co2 = pd.read_csv('/work/hawkins_lab/vulcan/data/egrid_state_2019.csv', header=1)
df_co2.columns = df_co2.columns.str.lower()

In [13]:
for file in file_list:
    df = gpd.read_parquet(file + '.parquet')
    df.rename(columns={'epa_id': 'geoid20'}, inplace=True)
    df.geoid20 = df.geoid20.astype(int)
    df['statefips'] = ((df.fips)/10**3).astype(int)
    df = pd.merge(df,df_co2,left_on="statefips", right_on="fipsst", how="left")
    df = pd.merge(df, df_climate, on="cbsa", how="left")
    # There is no climate data for 46102 because it didn't exist in that year with first valid row from above
    df = df.bfill()
    df = pd.merge(df, add_cbg_stats, on="geoid20", how="left")
    out_file = os.path.join(base_path,'parquet',file) + '_climate.parquet'
    df.to_parquet(out_file)
    print("Finished", file)
    print(df.describe())

Finished /work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_RES_epa
            geoid20    value_weig       objectid           cbsa      cbsa_pop  \
count  2.162300e+05  2.162300e+05  216230.000000  216230.000000  2.162300e+05   
mean   2.823702e+11  1.635471e+03  109195.969195   28103.100525  3.660384e+06   
std    1.569773e+11  1.503065e+03   62897.438788   13245.105056  5.257357e+06   
min    1.001020e+10  2.622073e-08       1.000000       1.110000  0.000000e+00   
25%    1.311713e+11  6.545606e+02   54603.250000   17460.000000  2.070500e+05   
50%    2.810595e+11  1.293124e+03  109574.500000   31080.000000  1.252890e+06   
75%    4.105100e+11  2.218838e+03  163672.750000   38300.000000  4.673634e+06   
max    5.604595e+11  1.689378e+05  217739.000000   49820.000000  1.931847e+07   

           cbsa_emp      cbsa_wrk      ac_total      ac_water       ac_land  \
count  2.162300e+05  2.162300e+05  2.162300e+05  2.162300e+05  2.162300e+05   
mean   1.716271e+06  1.667185e+06  9.218866